In [ ]:
import yfinance as yf
from datetime import date
import plotly.express as px

import pandas as pd
pd.options.plotting.backend = "plotly"

from DT_BCB_series import CarregaSerieBCB

data_inicial = '02/01/1995'
data_final   = '25/03/2021'


In [3]:

ibov = yf.download('^BVSP', start=pd.to_datetime(data_inicial, dayfirst=True), end=pd.to_datetime(data_final, dayfirst=True), auto_adjust=True)['Close']


[*********************100%***********************]  1 of 1 completed


In [10]:

dolar_ptax = CarregaSerieBCB(1, data_inicial, data_final)
ibov_dolar = ibov / dolar_ptax['valor']


TypeError: strptime() argument 1 must be str, not int

In [ ]:

df_ibov = pd.concat([ibov, ibov_dolar], axis=1)
df_ibov.columns = ['IBOV', 'IBOV_USD']

# IPCA
ipca = consulta_bc(433, data_inicial, data_final)
ipca['acumulado'] = (1 + ipca['valor'] / 100).cumprod().shift(1)
ipca['acumulado'].iloc[0] = 1

join = ipca.join(ibov, how='outer').fillna(method='ffill').dropna()
ibov_defla_ipca = join['Adj Close'] / join['acumulado']
df_ibov['IBOV_DEFLA_IPCA'] = ibov_defla_ipca

# IGPDI
igpdi = consulta_bc(190, data_inicial, data_final)
igpdi['acumulado'] = (1 + igpdi['valor'] / 100).cumprod().shift(1)
igpdi['acumulado'].iloc[0] = 1

join = igpdi.join(ibov, how='outer').fillna(method='ffill').dropna()
ibov_defla_igpdi = join['Adj Close'] / join['acumulado']
df_ibov['IBOV_DEFLA_IGPDI'] = ibov_defla_igpdi

In [ ]:
cdi = consulta_bc(12,data_inicial,data_final)
cdi.columns = ['diaria']
cdi['acumulado'] = (1 + cdi['diaria'] / 100).cumprod().shift(1)
cdi['acumulado'].iloc[0] = 1

df_ibov_cdi = pd.concat([ibov / ibov.iloc[0], cdi['acumulado']], axis=1).dropna()